# Cross validation organization and Preparation for all the subset



1. Components Only
2. Full Dataset
3. Missing Only
4. Non missing






In [ ]:
"/content/drive/MyDrive/PCB_MC/Data/components_only"
/content/drive/MyDrive/PCB_MC/Data/full_dataset
/content/drive/MyDrive/PCB_MC/Data/missing_only
/content/drive/MyDrive/PCB_MC/Data/non_missing

## Understanding COCO Format and Conversion Logic

COCO (Common Objects in Context) is a large-scale object detection, segmentation, and captioning dataset. Its annotation format is widely used in computer vision, especially for training deep learning models.

### COCO Annotation Structure
A COCO annotation file is a single JSON file that typically contains the following top-level keys:

*   **`info`**: General information about the dataset.
*   **`licenses`**: Information about the licenses under which the images are distributed.
*   **`categories`**: A list of object categories, each with an `id`, `name`, and `supercategory`.
*   **`images`**: A list of image metadata, each with an `id`, `width`, `height`, `file_name`, and optionally `license`, `flickr_url`, `coco_url`, `date_captured`.
*   **`annotations`**: A list of object annotations, each with an `id`, `image_id`, `category_id`, `bbox`, `area`, `iscrowd` (for segmentation, typically `0` for detection), and optionally `segmentation`.

### YOLO to COCO Bounding Box Conversion

YOLO format annotations are typically defined as: `class_id center_x center_y width height`.

These values are normalized relative to the image's width and height. To convert them to COCO's `[x_min, y_min, width, height]` format (absolute pixel values), we use the following formulas:

Given:
*   `img_width`, `img_height`: Dimensions of the image
*   `yolo_center_x`, `yolo_center_y`, `yolo_width`, `yolo_height`: Normalized YOLO coordinates (values between 0 and 1)

Conversion Steps:
1.  **Absolute `width` and `height`**:
    `coco_width = yolo_width * img_width`
    `coco_height = yolo_height * img_height`
2.  **Absolute `center_x` and `center_y`**:
    `abs_center_x = yolo_center_x * img_width`
    `abs_center_y = yolo_center_y * img_height`
3.  **Absolute `x_min` and `y_min`**:
    `coco_x_min = abs_center_x - (coco_width / 2)`
    `coco_y_min = abs_center_y - (coco_height / 2)`

The resulting COCO bounding box will be `[coco_x_min, coco_y_min, coco_width, coco_height]`.

The `area` for COCO is simply `coco_width * coco_height`.

### General Conversion Process

For each `fold_k`, and for each subset (`train`, `valid`, `test`):

1.  **Initialize COCO JSON structure**: Create empty lists for `images` and `annotations`.
2.  **Define Categories**: Manually define or load your object categories (e.g., `{'id': 1, 'name': 'component', 'supercategory': 'none'}`). This needs to be consistent across all folds.
3.  **Process Images and Labels**:
    *   Iterate through all image files in the current subset's `images` directory.
    *   For each image, read its dimensions (width, height).
    *   Add an entry to the `images` list with a unique `id`, `file_name`, `width`, and `height`.
    *   Find the corresponding YOLO label file (`.txt`) in the `labels` directory.
    *   Read each line in the YOLO label file.
    *   For each annotation in the label file, convert the YOLO `class_id` and bounding box to COCO format using the image dimensions.
    *   Add an entry to the `annotations` list with a unique `id`, `image_id`, `category_id`, `bbox`, and `area`.
4.  **Save COCO JSON**: Save the assembled COCO JSON structure to a file (e.g., `annotations_train.json`) in the respective fold's directory.

## Helper Functions for COCO Conversion

I'll now define two helper functions:
1.  `get_image_dimensions`: To get the width and height of an image.
2.  `yolo_to_coco_bbox`: To convert YOLO's normalized `center_x, center_y, width, height` format to COCO's absolute `x_min, y_min, width, height` format.

In [ ]:
def get_image_dimensions(image_path):
    try:
        with Image.open(image_path) as img:
            width, height = img.size
        return width, height
    except Exception as e:
        print(f"Error reading image {image_path}: {e}")
        return None, None

def yolo_to_coco_bbox(img_width, img_height, yolo_bbox_str):
    # yolo_bbox_str format: 'class_id center_x center_y width height'
    parts = yolo_bbox_str.split()

    # Ensure we have at least 5 parts for a valid YOLO bbox: class_id, center_x, center_y, width, height
    if len(parts) < 5:
        print(f"  Warning: Invalid YOLO format line: '{yolo_bbox_str.strip()}'. Expected at least 5 parts, got {len(parts)}. Skipping annotation.")
        return None, None, None # Return 3 Nones to match expected unpacking (class_id, bbox, area)

    try:
        class_id = int(parts[0])
        # Extract the relevant 4 bbox components. If there are extra components, ignore them.
        center_x_norm, center_y_norm, width_norm, height_norm = map(float, parts[1:5])
    except ValueError as e:
        print(f"  Warning: Could not parse YOLO bbox components from line: '{yolo_bbox_str.strip()}'. Error: {e}. Skipping annotation.")
        return None, None, None # Return 3 Nones for parsing errors

    # Convert normalized to absolute values
    abs_center_x = center_x_norm * img_width
    abs_center_y = center_y_norm * img_height
    coco_width = width_norm * img_width
    coco_height = height_norm * img_height

    # Calculate top-left corner (x_min, y_min)
    coco_x_min = abs_center_x - (coco_width / 2)
    coco_y_min = abs_center_y - (coco_height / 2)

    # COCO bbox format: [x_min, y_min, width, height]
    bbox = [coco_x_min, coco_y_min, coco_width, coco_height]
    area = coco_width * coco_height

    return class_id, bbox, area

##Subset: Components Only

In [ ]:
import os
import json
import cv2
from PIL import Image

# Base path for the kfold data
kfold_base_path = '/content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data'

# Define your categories. Assuming a single 'component' class based on the problem description.
# You might need to adjust this if you have multiple classes.
categories=[{"id":0,"name":"printed-circuit-board-cgnF-H8Rs","supercategory":"none"},
 {"id":1,"name":"Button","supercategory":"printed-circuit-board-cgnF-H8Rs"},
  {"id":2,"name":"Capacitor","supercategory":"printed-circuit-board-cgnF-H8Rs"},
   {"id":3,"name":"Clock","supercategory":"printed-circuit-board-cgnF-H8Rs"},
    {"id":4,"name":"Connector","supercategory":"printed-circuit-board-cgnF-H8Rs"},
     {"id":5,"name":"Diode","supercategory":"printed-circuit-board-cgnF-H8Rs"},
      {"id":6,"name":"Display","supercategory":"printed-circuit-board-cgnF-H8Rs"},
       {"id":7,"name":"EM","supercategory":"printed-circuit-board-cgnF-H8Rs"},
        {"id":8,"name":"Electrolytic Capacitor","supercategory":"printed-circuit-board-cgnF-H8Rs"},
         {"id":9,"name":"Ferrite Bead","supercategory":"printed-circuit-board-cgnF-H8Rs"},
          {"id":10,"name":"Fuse","supercategory":"printed-circuit-board-cgnF-H8Rs"},
           {"id":11,"name":"Heatsink","supercategory":"printed-circuit-board-cgnF-H8Rs"},
            {"id":12,"name":"IC","supercategory":"printed-circuit-board-cgnF-H8Rs"},
             {"id":13,"name":"Inductor","supercategory":"printed-circuit-board-cgnF-H8Rs"},
              {"id":14,"name":"Jumper","supercategory":"printed-circuit-board-cgnF-H8Rs"},
               {"id":15,"name":"Led","supercategory":"printed-circuit-board-cgnF-H8Rs"},
                {"id":16,"name":"Pads","supercategory":"printed-circuit-board-cgnF-H8Rs"},
                 {"id":17,"name":"Pins","supercategory":"printed-circuit-board-cgnF-H8Rs"},
                  {"id":18,"name":"Potentiometer","supercategory":"printed-circuit-board-cgnF-H8Rs"},
                   {"id":19,"name":"Resistor","supercategory":"printed-circuit-board-cgnF-H8Rs"},
                    {"id":20,"name":"Switch","supercategory":"printed-circuit-board-cgnF-H8Rs"},
                     {"id":21,"name":"Test Point","supercategory":"printed-circuit-board-cgnF-H8Rs"},
                      {"id":22,"name":"Transistor","supercategory":"printed-circuit-board-cgnF-H8Rs"},
                       {"id":23,"name":"Zener Diode","supercategory":"printed-circuit-board-cgnF-H8Rs"}]

print(f"K-fold base path set to: {kfold_base_path}")
print(f"Defined COCO categories: {categories}")

K-fold base path set to: /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data
Defined COCO categories: [{'id': 0, 'name': 'printed-circuit-board-cgnF-H8Rs', 'supercategory': 'none'}, {'id': 1, 'name': 'Button', 'supercategory': 'printed-circuit-board-cgnF-H8Rs'}, {'id': 2, 'name': 'Capacitor', 'supercategory': 'printed-circuit-board-cgnF-H8Rs'}, {'id': 3, 'name': 'Clock', 'supercategory': 'printed-circuit-board-cgnF-H8Rs'}, {'id': 4, 'name': 'Connector', 'supercategory': 'printed-circuit-board-cgnF-H8Rs'}, {'id': 5, 'name': 'Diode', 'supercategory': 'printed-circuit-board-cgnF-H8Rs'}, {'id': 6, 'name': 'Display', 'supercategory': 'printed-circuit-board-cgnF-H8Rs'}, {'id': 7, 'name': 'EM', 'supercategory': 'printed-circuit-board-cgnF-H8Rs'}, {'id': 8, 'name': 'Electrolytic Capacitor', 'supercategory': 'printed-circuit-board-cgnF-H8Rs'}, {'id': 9, 'name': 'Ferrite Bead', 'supercategory': 'printed-circuit-board-cgnF-H8Rs'}, {'id': 10, 'name': 'Fuse', 'supercategory': 'printed-cir

In [ ]:
# Iterate through all folds from 0 to 4
for fold_idx in range(5):
    print(f"\n--- Processing Fold {fold_idx} ---")
    for subset in ['train', 'valid']:
        print(f"\nProcessing fold_{fold_idx}/{subset}...")

        fold_subset_path = os.path.join(kfold_base_path, f'fold_{fold_idx}', subset)
        images_path = os.path.join(fold_subset_path, 'images')
        labels_path = os.path.join(fold_subset_path, 'labels')

        coco_output = {
            "info": {},
            "licenses": [],
            "categories": categories,
            "images": [],
            "annotations": []
        }

        image_id_counter = 0
        annotation_id_counter = 0

        if not os.path.exists(images_path):
            print(f"  Images path not found: {images_path}. Skipping.")
            continue

        for img_filename in os.listdir(images_path):
            if img_filename.endswith(('.jpg', '.jpeg', '.png')):
                image_path = os.path.join(images_path, img_filename)
                img_width, img_height = get_image_dimensions(image_path)

                if img_width is None or img_height is None:
                    continue

                # Add image info to COCO structure
                image_id = image_id_counter
                coco_output['images'].append({
                    "id": image_id,
                    "width": img_width,
                    "height": img_height,
                    "file_name": img_filename,
                    "license": 0,
                    "flickr_url": "",
                    "coco_url": "",
                    "date_captured": ""
                })
                image_id_counter += 1

                # Process corresponding YOLO label file
                label_filename = os.path.splitext(img_filename)[0] + '.txt'
                label_file_path = os.path.join(labels_path, label_filename)

                if os.path.exists(label_file_path):
                    with open(label_file_path, 'r') as f:
                        for line in f.readlines():
                            class_id, bbox, area = yolo_to_coco_bbox(img_width, img_height, line.strip())
                            if bbox:
                                # Add annotation info to COCO structure
                                coco_output['annotations'].append({
                                    "id": annotation_id_counter,
                                    "image_id": image_id,
                                    "category_id": class_id, # Assuming YOLO class_id maps directly to COCO category_id
                                    "bbox": [round(coord) for coord in bbox], # Round to nearest integer
                                    "area": round(area),
                                    "iscrowd": 0
                                })
                                annotation_id_counter += 1
                else:
                    print(f"  Warning: Label file not found for {img_filename} at {label_file_path}")

        # Save the COCO JSON file
        output_json_path = os.path.join(fold_subset_path, 'annotations.json')
        with open(output_json_path, 'w') as f:
            json.dump(coco_output, f, indent=4)

        print(f"  COCO annotations saved to: {output_json_path}")
        print(f"  Total images in {subset}: {len(coco_output['images'])}")
        print(f"  Total annotations in {subset}: {len(coco_output['annotations'])}")

print("\nAll K-fold COCO conversions complete.")


--- Processing Fold 0 ---

Processing fold_0/train...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/train/annotations.json
  Total images in train: 492
  Total annotations in train: 93885

Processing fold_0/valid...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_0/valid/annotations.json
  Total images in valid: 123
  Total annotations in valid: 23204

--- Processing Fold 1 ---

Processing fold_1/train...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_1/train/annotations.json
  Total images in train: 492
  Total annotations in train: 95920

Processing fold_1/valid...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/components_only/kfold_data/fold_1/valid/annotations.json
  Total images in valid: 123
  Total annotations in valid: 21169

--- Processing Fold 2 ---

Processing fold_2/train...
  COCO annotations saved to: /conte

##Subset: Full Dataset

In [ ]:
import os
import json
import cv2
from PIL import Image

# Base path for the kfold data
kfold_base_path = '/content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data'

# Define your categories. Assuming a single 'component' class based on the problem description.
# You might need to adjust this if you have multiple classes.
categories=[
 {"id":1,"name":"Button","supercategory":"none"},
  {"id":2,"name":"Capacitor","supercategory":"none"},
   {"id":3,"name":"Clock","supercategory":"none"},
    {"id":4,"name":"Connector","supercategory":"none"},
     {"id":5,"name":"Diode","supercategory":"none"},
      {"id":6,"name":"Display","supercategory":"none"},
       {"id":7,"name":"EM","supercategory":"none"},
        {"id":8,"name":"Electrolytic Capacitor","supercategory":"none"},
         {"id":9,"name":"Ferrite Bead","supercategory":"none"},
          {"id":10,"name":"Fuse","supercategory":"none"},
           {"id":11,"name":"Heatsink","supercategory":"none"},
            {"id":12,"name":"IC","supercategory":"none"},
             {"id":13,"name":"Inductor","supercategory":"none"},
              {"id":14,"name":"Jumper","supercategory":"none"},
               {"id":15,"name":"Led","supercategory":"none"},
                {"id":16,"name":"Missing Capacitor","supercategory":"none"},
                 {"id":17,"name":"Missing Component","supercategory":"none"},
                  {"id":18,"name":"Missing Diode","supercategory":"none"},
                   {"id":19,"name":"Missing Ferrite bead","supercategory":"none"},
                    {"id":20,"name":"Missing IC","supercategory":"none"},
                     {"id":21,"name":"Missing Inductor","supercategory":"none"},
                      {"id":22,"name":"Missing Led","supercategory":"none"},
                       {"id":23,"name":"Missing Resistor","supercategory":"none"},
                        {"id":24,"name":"Pads","supercategory":"none"},
                         {"id":25,"name":"Pins","supercategory":"none"},
                          {"id":26,"name":"Potentiometer","supercategory":"none"},
                           {"id":27,"name":"Resistor","supercategory":"none"},
                            {"id":28,"name":"Switch","supercategory":"none"},
                             {"id":29,"name":"Test Point","supercategory":"none"},
                              {"id":30,"name":"Transistor","supercategory":"none"},
                               {"id":31,"name":"Zener Diode","supercategory":"none"}]


print(f"K-fold base path set to: {kfold_base_path}")
print(f"Defined COCO categories: {categories}")

K-fold base path set to: /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data
Defined COCO categories: [{'id': 1, 'name': 'Button', 'supercategory': 'none'}, {'id': 2, 'name': 'Capacitor', 'supercategory': 'none'}, {'id': 3, 'name': 'Clock', 'supercategory': 'none'}, {'id': 4, 'name': 'Connector', 'supercategory': 'none'}, {'id': 5, 'name': 'Diode', 'supercategory': 'none'}, {'id': 6, 'name': 'Display', 'supercategory': 'none'}, {'id': 7, 'name': 'EM', 'supercategory': 'none'}, {'id': 8, 'name': 'Electrolytic Capacitor', 'supercategory': 'none'}, {'id': 9, 'name': 'Ferrite Bead', 'supercategory': 'none'}, {'id': 10, 'name': 'Fuse', 'supercategory': 'none'}, {'id': 11, 'name': 'Heatsink', 'supercategory': 'none'}, {'id': 12, 'name': 'IC', 'supercategory': 'none'}, {'id': 13, 'name': 'Inductor', 'supercategory': 'none'}, {'id': 14, 'name': 'Jumper', 'supercategory': 'none'}, {'id': 15, 'name': 'Led', 'supercategory': 'none'}, {'id': 16, 'name': 'Missing Capacitor', 'supercategory':

In [ ]:
# Iterate through all folds from 0 to 4
for fold_idx in range(5):
    print(f"\n--- Processing Fold {fold_idx} ---")
    for subset in ['train', 'valid']:
        print(f"\nProcessing fold_{fold_idx}/{subset}...")

        fold_subset_path = os.path.join(kfold_base_path, f'fold_{fold_idx}', subset)
        images_path = os.path.join(fold_subset_path, 'images')
        labels_path = os.path.join(fold_subset_path, 'labels')

        coco_output = {
            "info": {},
            "licenses": [],
            "categories": categories,
            "images": [],
            "annotations": []
        }

        image_id_counter = 0
        annotation_id_counter = 0

        if not os.path.exists(images_path):
            print(f"  Images path not found: {images_path}. Skipping.")
            continue

        for img_filename in os.listdir(images_path):
            if img_filename.endswith(('.jpg', '.jpeg', '.png')):
                image_path = os.path.join(images_path, img_filename)
                img_width, img_height = get_image_dimensions(image_path)

                if img_width is None or img_height is None:
                    continue

                # Add image info to COCO structure
                image_id = image_id_counter
                coco_output['images'].append({
                    "id": image_id,
                    "width": img_width,
                    "height": img_height,
                    "file_name": img_filename,
                    "license": 0,
                    "flickr_url": "",
                    "coco_url": "",
                    "date_captured": ""
                })
                image_id_counter += 1

                # Process corresponding YOLO label file
                label_filename = os.path.splitext(img_filename)[0] + '.txt'
                label_file_path = os.path.join(labels_path, label_filename)

                if os.path.exists(label_file_path):
                    with open(label_file_path, 'r') as f:
                        for line in f.readlines():
                            class_id, bbox, area = yolo_to_coco_bbox(img_width, img_height, line.strip())
                            if bbox:
                                # Add annotation info to COCO structure
                                coco_output['annotations'].append({
                                    "id": annotation_id_counter,
                                    "image_id": image_id,
                                    "category_id": class_id, # Assuming YOLO class_id maps directly to COCO category_id
                                    "bbox": [round(coord) for coord in bbox], # Round to nearest integer
                                    "area": round(area),
                                    "iscrowd": 0
                                })
                                annotation_id_counter += 1
                else:
                    print(f"  Warning: Label file not found for {img_filename} at {label_file_path}")

        # Save the COCO JSON file
        output_json_path = os.path.join(fold_subset_path, 'annotations.json')
        with open(output_json_path, 'w') as f:
            json.dump(coco_output, f, indent=4)

        print(f"  COCO annotations saved to: {output_json_path}")
        print(f"  Total images in {subset}: {len(coco_output['images'])}")
        print(f"  Total annotations in {subset}: {len(coco_output['annotations'])}")

print("\nAll K-fold COCO conversions complete.")


--- Processing Fold 0 ---

Processing fold_0/train...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/train/annotations.json
  Total images in train: 492
  Total annotations in train: 101078

Processing fold_0/valid...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_0/valid/annotations.json
  Total images in valid: 123
  Total annotations in valid: 25446

--- Processing Fold 1 ---

Processing fold_1/train...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_1/train/annotations.json
  Total images in train: 492
  Total annotations in train: 103610

Processing fold_1/valid...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/full_dataset/kfold_data/fold_1/valid/annotations.json
  Total images in valid: 123
  Total annotations in valid: 22914

--- Processing Fold 2 ---

Processing fold_2/train...
  COCO annotations saved to: /content/drive/M

##Subset: Missing Only

In [ ]:
import os
import json
import cv2
from PIL import Image

# Base path for the kfold data
kfold_base_path = '/content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data'

# Define your categories. Assuming a single 'component' class based on the problem description.
# You might need to adjust this if you have multiple classes.
categories=[
 {"id":1,"name":"Missing Capacitor","supercategory":"none"},
  {"id":2,"name":"Missing Component","supercategory":"none"},
   {"id":3,"name":"Missing Diode","supercategory":"none"},
    {"id":4,"name":"Missing Ferrite bead","supercategory":"none"},
     {"id":5,"name":"Missing IC","supercategory":"none"},
      {"id":6,"name":"Missing Inductor","supercategory":"none"},
       {"id":7,"name":"Missing Led","supercategory":"none"},
        {"id":8,"name":"Missing Resistor","supercategory":"none"}]

print(f"K-fold base path set to: {kfold_base_path}")
print(f"Defined COCO categories: {categories}")

K-fold base path set to: /content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data
Defined COCO categories: [{'id': 1, 'name': 'Missing Capacitor', 'supercategory': 'none'}, {'id': 2, 'name': 'Missing Component', 'supercategory': 'none'}, {'id': 3, 'name': 'Missing Diode', 'supercategory': 'none'}, {'id': 4, 'name': 'Missing Ferrite bead', 'supercategory': 'none'}, {'id': 5, 'name': 'Missing IC', 'supercategory': 'none'}, {'id': 6, 'name': 'Missing Inductor', 'supercategory': 'none'}, {'id': 7, 'name': 'Missing Led', 'supercategory': 'none'}, {'id': 8, 'name': 'Missing Resistor', 'supercategory': 'none'}]


In [ ]:
# Iterate through all folds from 0 to 4
for fold_idx in range(5):
    print(f"\n--- Processing Fold {fold_idx} ---")
    for subset in ['train', 'valid']:
        print(f"\nProcessing fold_{fold_idx}/{subset}...")

        fold_subset_path = os.path.join(kfold_base_path, f'fold_{fold_idx}', subset)
        images_path = os.path.join(fold_subset_path, 'images')
        labels_path = os.path.join(fold_subset_path, 'labels')

        coco_output = {
            "info": {},
            "licenses": [],
            "categories": categories,
            "images": [],
            "annotations": []
        }

        image_id_counter = 0
        annotation_id_counter = 0

        if not os.path.exists(images_path):
            print(f"  Images path not found: {images_path}. Skipping.")
            continue

        for img_filename in os.listdir(images_path):
            if img_filename.endswith(('.jpg', '.jpeg', '.png')):
                image_path = os.path.join(images_path, img_filename)
                img_width, img_height = get_image_dimensions(image_path)

                if img_width is None or img_height is None:
                    continue

                # Add image info to COCO structure
                image_id = image_id_counter
                coco_output['images'].append({
                    "id": image_id,
                    "width": img_width,
                    "height": img_height,
                    "file_name": img_filename,
                    "license": 0,
                    "flickr_url": "",
                    "coco_url": "",
                    "date_captured": ""
                })
                image_id_counter += 1

                # Process corresponding YOLO label file
                label_filename = os.path.splitext(img_filename)[0] + '.txt'
                label_file_path = os.path.join(labels_path, label_filename)

                if os.path.exists(label_file_path):
                    with open(label_file_path, 'r') as f:
                        for line in f.readlines():
                            class_id, bbox, area = yolo_to_coco_bbox(img_width, img_height, line.strip())
                            if bbox:
                                # Add annotation info to COCO structure
                                coco_output['annotations'].append({
                                    "id": annotation_id_counter,
                                    "image_id": image_id,
                                    "category_id": class_id, # Assuming YOLO class_id maps directly to COCO category_id
                                    "bbox": [round(coord) for coord in bbox], # Round to nearest integer
                                    "area": round(area),
                                    "iscrowd": 0
                                })
                                annotation_id_counter += 1
                else:
                    print(f"  Warning: Label file not found for {img_filename} at {label_file_path}")

        # Save the COCO JSON file
        output_json_path = os.path.join(fold_subset_path, 'annotations.json')
        with open(output_json_path, 'w') as f:
            json.dump(coco_output, f, indent=4)

        print(f"  COCO annotations saved to: {output_json_path}")
        print(f"  Total images in {subset}: {len(coco_output['images'])}")
        print(f"  Total annotations in {subset}: {len(coco_output['annotations'])}")

print("\nAll K-fold COCO conversions complete.")


--- Processing Fold 0 ---

Processing fold_0/train...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/train/annotations.json
  Total images in train: 234
  Total annotations in train: 7566

Processing fold_0/valid...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_0/valid/annotations.json
  Total images in valid: 59
  Total annotations in valid: 1868

--- Processing Fold 1 ---

Processing fold_1/train...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_1/train/annotations.json
  Total images in train: 234
  Total annotations in train: 7736

Processing fold_1/valid...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/missing_only/kfold_data/fold_1/valid/annotations.json
  Total images in valid: 59
  Total annotations in valid: 1698

--- Processing Fold 2 ---

Processing fold_2/train...
  COCO annotations saved to: /content/drive/MyDrive/P

##Subset: Non Missing

In [ ]:
import os
import json
import cv2
from PIL import Image

# Base path for the kfold data
kfold_base_path = '/content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data'

# Define your categories. Assuming a single 'component' class based on the problem description.
# You might need to adjust this if you have multiple classes.
categories=[
 {"id":1,"name":"Button","supercategory":"none"},
  {"id":2,"name":"Capacitor","supercategory":"none"},
   {"id":3,"name":"Clock","supercategory":"none"},
    {"id":4,"name":"Connector","supercategory":"none"},
     {"id":5,"name":"Diode","supercategory":"none"},
      {"id":6,"name":"Display","supercategory":"none"},
       {"id":7,"name":"EM","supercategory":"none"},
        {"id":8,"name":"Electrolytic Capacitor","supercategory":"none"},
         {"id":9,"name":"Ferrite Bead","supercategory":"none"},
          {"id":10,"name":"Fuse","supercategory":"none"},
           {"id":11,"name":"Heatsink","supercategory":"none"},
            {"id":12,"name":"IC","supercategory":"none"},
             {"id":13,"name":"Inductor","supercategory":"none"},
              {"id":14,"name":"Jumper","supercategory":"none"},
               {"id":15,"name":"Led","supercategory":"none"},
                {"id":16,"name":"Pads","supercategory":"none"},
                 {"id":17,"name":"Pins","supercategory":"none"},
                  {"id":18,"name":"Potentiometer","supercategory":"none"},
                   {"id":19,"name":"Resistor","supercategory":"none"},
                    {"id":20,"name":"Switch","supercategory":"none"},
                     {"id":21,"name":"Test Point","supercategory":"none"},
                      {"id":22,"name":"Transistor","supercategory":"none"},
                       {"id":23,"name":"Zener Diode","supercategory":"none"}]

print(f"K-fold base path set to: {kfold_base_path}")
print(f"Defined COCO categories: {categories}")

K-fold base path set to: /content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data
Defined COCO categories: [{'id': 1, 'name': 'Button', 'supercategory': 'none'}, {'id': 2, 'name': 'Capacitor', 'supercategory': 'none'}, {'id': 3, 'name': 'Clock', 'supercategory': 'none'}, {'id': 4, 'name': 'Connector', 'supercategory': 'none'}, {'id': 5, 'name': 'Diode', 'supercategory': 'none'}, {'id': 6, 'name': 'Display', 'supercategory': 'none'}, {'id': 7, 'name': 'EM', 'supercategory': 'none'}, {'id': 8, 'name': 'Electrolytic Capacitor', 'supercategory': 'none'}, {'id': 9, 'name': 'Ferrite Bead', 'supercategory': 'none'}, {'id': 10, 'name': 'Fuse', 'supercategory': 'none'}, {'id': 11, 'name': 'Heatsink', 'supercategory': 'none'}, {'id': 12, 'name': 'IC', 'supercategory': 'none'}, {'id': 13, 'name': 'Inductor', 'supercategory': 'none'}, {'id': 14, 'name': 'Jumper', 'supercategory': 'none'}, {'id': 15, 'name': 'Led', 'supercategory': 'none'}, {'id': 16, 'name': 'Pads', 'supercategory': 'none'}, {'id

In [ ]:
# Iterate through all folds from 0 to 4
for fold_idx in range(5):
    print(f"\n--- Processing Fold {fold_idx} ---")
    for subset in ['train', 'valid']:
        print(f"\nProcessing fold_{fold_idx}/{subset}...")

        fold_subset_path = os.path.join(kfold_base_path, f'fold_{fold_idx}', subset)
        images_path = os.path.join(fold_subset_path, 'images')
        labels_path = os.path.join(fold_subset_path, 'labels')

        coco_output = {
            "info": {},
            "licenses": [],
            "categories": categories,
            "images": [],
            "annotations": []
        }

        image_id_counter = 0
        annotation_id_counter = 0

        if not os.path.exists(images_path):
            print(f"  Images path not found: {images_path}. Skipping.")
            continue

        for img_filename in os.listdir(images_path):
            if img_filename.endswith(('.jpg', '.jpeg', '.png')):
                image_path = os.path.join(images_path, img_filename)
                img_width, img_height = get_image_dimensions(image_path)

                if img_width is None or img_height is None:
                    continue

                # Add image info to COCO structure
                image_id = image_id_counter
                coco_output['images'].append({
                    "id": image_id,
                    "width": img_width,
                    "height": img_height,
                    "file_name": img_filename,
                    "license": 0,
                    "flickr_url": "",
                    "coco_url": "",
                    "date_captured": ""
                })
                image_id_counter += 1

                # Process corresponding YOLO label file
                label_filename = os.path.splitext(img_filename)[0] + '.txt'
                label_file_path = os.path.join(labels_path, label_filename)

                if os.path.exists(label_file_path):
                    with open(label_file_path, 'r') as f:
                        for line in f.readlines():
                            class_id, bbox, area = yolo_to_coco_bbox(img_width, img_height, line.strip())
                            if bbox:
                                # Add annotation info to COCO structure
                                coco_output['annotations'].append({
                                    "id": annotation_id_counter,
                                    "image_id": image_id,
                                    "category_id": class_id, # Assuming YOLO class_id maps directly to COCO category_id
                                    "bbox": [round(coord) for coord in bbox], # Round to nearest integer
                                    "area": round(area),
                                    "iscrowd": 0
                                })
                                annotation_id_counter += 1
                else:
                    print(f"  Warning: Label file not found for {img_filename} at {label_file_path}")

        # Save the COCO JSON file
        output_json_path = os.path.join(fold_subset_path, 'annotations.json')
        with open(output_json_path, 'w') as f:
            json.dump(coco_output, f, indent=4)

        print(f"  COCO annotations saved to: {output_json_path}")
        print(f"  Total images in {subset}: {len(coco_output['images'])}")
        print(f"  Total annotations in {subset}: {len(coco_output['annotations'])}")

print("\nAll K-fold COCO conversions complete.")


--- Processing Fold 0 ---

Processing fold_0/train...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/train/annotations.json
  Total images in train: 257
  Total annotations in train: 45929

Processing fold_0/valid...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_0/valid/annotations.json
  Total images in valid: 65
  Total annotations in valid: 12846

--- Processing Fold 1 ---

Processing fold_1/train...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_1/train/annotations.json
  Total images in train: 257
  Total annotations in train: 48889

Processing fold_1/valid...
  COCO annotations saved to: /content/drive/MyDrive/PCB_MC/Data/non_missing/kfold_data/fold_1/valid/annotations.json
  Total images in valid: 65
  Total annotations in valid: 9886

--- Processing Fold 2 ---

Processing fold_2/train...
  COCO annotations saved to: /content/drive/MyDrive/PC